# Plano de Implementação — TCC Cartola FC
## Predição e Recomendação de Escalações com Orçamento Dinâmico

**FGV/EMAp** | Gabriel Rodrigues de Deus Abad | Orientador: Prof. Moacyr Silva

---

### Visão Geral do Pipeline

```
df_features.csv (gerado pelo repositório)
        │
        ├── Notebook 1: train_eval.ipynb       → walk-forward (rodadas 5–18)
        │       ├── Modelos por posição
        │       ├── Otimizador PLI
        │       ├── Orçamento Dinâmico
        │       └── Outputs: resultados_walkforward.csv
        │                    historico_orcamento.csv
        │                    historico_escalacoes.csv
        │
        └── Notebook 2: previsao_segundo_turno.ipynb → rodadas 19–36
                ├── Treina em tudo (rodadas 1–18)
                ├── Coleta API incremental
                └── Output: escalacoes_segundo_turno.csv
```

### Dois Cenários de Avaliação

| Cenário | Rodadas | Objetivo | Output principal |
|---|---|---|---|
| **A — Retrospectivo** | 5–18 | Métricas quantitativas (RMSE, Spearman, eficiência) | Tabelas e gráficos do TCC |
| **B — Prospectivo** | 19–36 | Escalações reais do segundo turno | Produto final do sistema |


## 0. Imports e Configuração Global

In [ ]:

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Modelos
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
import xgboost as xgb
import lightgbm as lgb

# Otimização
from pulp import (LpProblem, LpVariable, LpMaximize, LpBinary,
                  lpSum, PULP_CBC_CMD, value)

# Métricas
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Otimização de hiperparâmetros
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Visualização
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns

print("Imports OK")


In [ ]:

# ─── Configuração Global ───────────────────────────────────────────────────────

# Caminhos — ajuste conforme seu ambiente
PATH_FEATURES  = 'df_features.csv'
PATH_PLAYERS   = 'players.csv'
PATH_GAMES     = 'games.csv'
PATH_OUT       = 'outputs/'        # pasta onde os CSVs de resultado serão salvos

import os
os.makedirs(PATH_OUT, exist_ok=True)

# Posições
POS_MAP = {1: 'Goleiro', 2: 'Lateral', 3: 'Zagueiro',
           4: 'Meia',    5: 'Atacante', 6: 'Técnico'}
VAGAS_POS = {1: 1, 2: 2, 3: 2, 4: 3, 5: 3, 6: 1}   # restrições Cartola

# Protocolo walk-forward
MIN_TRAIN_ROUND = 4    # primeira rodada com features completas (janela 3r)
MIN_TEST_ROUND  = 5    # primeira rodada que prevemos
N_ROUNDS        = 18   # total de rodadas disponíveis

# Orçamento simulado
ORCAMENTO_INICIAL = 130.0   # cartoletas no início da rodada 5
MAX_CLUBE         = 5       # máximo de atletas do mesmo clube

# Modelos a comparar (chave = nome no output)
MODELOS = ['baseline', 'ridge', 'xgboost', 'lightgbm']

# Features que NÃO entram no modelo (identificadores + target)
COLS_DROP = [
    'rodada_id', 'atleta_id', 'apelido', 'posicao_id', 'clube_id',
    'adversario_id', 'pontos_num',
    # h2h tem >95% NaN nas rodadas 1-18; será imputado antes
]

print("Configuração OK")


## 1. Carrega e Prepara os Dados

In [ ]:

df = pd.read_csv(PATH_FEATURES)
players = pd.read_csv(PATH_PLAYERS)
games   = pd.read_csv(PATH_GAMES)

print(f"df_features : {df.shape}")
print(f"players     : {players.shape}")
print(f"games       : {games.shape}")
print(f"Rodadas em df_features: {sorted(df['rodada_id'].unique())}")
df.head(3)


In [ ]:

# ─── Pré-processamento do df_features ─────────────────────────────────────────

# 1. Imputa h2h_pts_avg com a média histórica do atleta (evita dropar a feature)
df['h2h_pts_avg'] = df.groupby('atleta_id')['h2h_pts_avg'].transform(
    lambda x: x.fillna(x.mean())
)
# Atletas sem nenhum h2h histórico: imputa com média global
df['h2h_pts_avg'] = df['h2h_pts_avg'].fillna(df['h2h_pts_avg'].mean())

# 2. Lista final de features (tudo que não é identificador/target)
FEATURE_COLS = [c for c in df.columns if c not in COLS_DROP]

print(f"Features utilizadas: {len(FEATURE_COLS)}")
print(f"NaNs restantes por feature:\n{df[FEATURE_COLS].isnull().sum()[df[FEATURE_COLS].isnull().sum() > 0]}")


## 2. Otimizador PLI de Escalação

O otimizador recebe um DataFrame com predições e retorna a escalação de maior
pontuação esperada respeitando todas as restrições do Cartola FC.


In [ ]:

def otimiza_escalacao(df_pred, orcamento, max_clube=MAX_CLUBE, vagas_pos=VAGAS_POS):
    """
    Resolve o PLI de escalação do Cartola FC.

    Parâmetros
    ----------
    df_pred   : DataFrame com colunas [atleta_id, posicao_id, clube_id,
                                        preco_num, y_hat, apelido]
    orcamento : float — cartoletas disponíveis
    max_clube : int   — máximo de atletas do mesmo clube (padrão 5)
    vagas_pos : dict  — {posicao_id: n_vagas}

    Retorna
    -------
    DataFrame com os 12 atletas escalados, ou None se infeasível.
    """
    df_pred = df_pred.copy().reset_index(drop=True)
    atletas = df_pred.index.tolist()

    prob = LpProblem("cartola_escalacao", LpMaximize)
    x    = {i: LpVariable(f"x_{i}", cat=LpBinary) for i in atletas}

    # Objetivo: maximizar pontuação esperada total
    prob += lpSum(df_pred.loc[i, 'y_hat'] * x[i] for i in atletas)

    # Restrição de orçamento
    prob += lpSum(df_pred.loc[i, 'preco_num'] * x[i] for i in atletas) <= orcamento

    # Restrições de posição
    for pos, n in vagas_pos.items():
        atletas_pos = [i for i in atletas if df_pred.loc[i, 'posicao_id'] == pos]
        prob += lpSum(x[i] for i in atletas_pos) == n

    # Restrição de concentração por clube
    for clube in df_pred['clube_id'].unique():
        atletas_clube = [i for i in atletas if df_pred.loc[i, 'clube_id'] == clube]
        prob += lpSum(x[i] for i in atletas_clube) <= max_clube

    prob.solve(PULP_CBC_CMD(msg=0))

    if prob.status != 1:
        print(f"  [AVISO] PLI infeasível — status {prob.status}")
        return None

    selecionados = [i for i in atletas if value(x[i]) > 0.5]
    escalacao    = df_pred.loc[selecionados].copy()

    # Define capitão: maior razão y_hat / (std_pts_5r + 1e-6)  →  Sharpe de pontuação
    if 'pts_std_5r' in escalacao.columns:
        escalacao['sharpe'] = escalacao['y_hat'] / (escalacao['pts_std_5r'] + 1e-6)
    else:
        escalacao['sharpe'] = escalacao['y_hat']
    escalacao['capitao'] = escalacao['sharpe'] == escalacao['sharpe'].max()

    return escalacao.drop(columns=['sharpe'])


def escalacao_otima(df_real, orcamento, max_clube=MAX_CLUBE, vagas_pos=VAGAS_POS):
    """
    Calcula a escalação ótima usando pontuações reais (hindsight).
    Usa a mesma função, mas com y_hat = pontos_num real.
    """
    df_tmp = df_real.copy()
    df_tmp['y_hat'] = df_tmp['pontos_num']
    return otimiza_escalacao(df_tmp, orcamento, max_clube, vagas_pos)


print("Otimizador PLI definido ✓")


## 3. Modelos Preditivos por Posição

Cada posição é tratada como um problema de regressão independente.
O modelo recebe as features do atleta na rodada t-1 e prevê pontos_num em t.


In [ ]:

def baseline_predict(df_train, df_test):
    """Média das últimas 5 rodadas jogadas pelo atleta (pts_avg_5r)."""
    return df_test['pts_avg_5r'].fillna(df_test['pts_avg_3r']).fillna(
           df_test['media_num'] if 'media_num' in df_test.columns else 0)


def treina_ridge(X_train, y_train):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model',  Ridge())
    ])
    tscv   = TimeSeriesSplit(n_splits=3)
    alphas = [0.01, 0.1, 1, 10, 100]
    best_alpha, best_score = 1.0, np.inf
    for a in alphas:
        pipe.set_params(model__alpha=a)
        scores = -cross_val_score(pipe, X_train, y_train,
                                  cv=tscv, scoring='neg_mean_absolute_error')
        if scores.mean() < best_score:
            best_score, best_alpha = scores.mean(), a
    pipe.set_params(model__alpha=best_alpha)
    pipe.fit(X_train, y_train)
    return pipe


def treina_xgboost(X_train, y_train, n_trials=30):
    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 50, 400),
            'max_depth':        trial.suggest_int('max_depth', 2, 7),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
            'random_state':     42,
            'n_jobs':           -1,
        }
        model  = xgb.XGBRegressor(**params, verbosity=0)
        tscv   = TimeSeriesSplit(n_splits=3)
        scores = -cross_val_score(model, X_train, y_train,
                                  cv=tscv, scoring='neg_mean_absolute_error')
        return scores.mean()

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = xgb.XGBRegressor(**study.best_params, verbosity=0, random_state=42, n_jobs=-1)
    best.fit(X_train, y_train)
    return best


def treina_lightgbm(X_train, y_train, n_trials=30):
    def objective(trial):
        params = {
            'n_estimators':    trial.suggest_int('n_estimators', 50, 400),
            'max_depth':       trial.suggest_int('max_depth', 2, 7),
            'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'num_leaves':      trial.suggest_int('num_leaves', 10, 100),
            'subsample':       trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree':trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha':       trial.suggest_float('reg_alpha', 1e-4, 10, log=True),
            'random_state':    42,
            'n_jobs':          -1,
            'verbose':         -1,
        }
        model  = lgb.LGBMRegressor(**params)
        tscv   = TimeSeriesSplit(n_splits=3)
        scores = -cross_val_score(model, X_train, y_train,
                                  cv=tscv, scoring='neg_mean_absolute_error')
        return scores.mean()

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    best = lgb.LGBMRegressor(**study.best_params, random_state=42, n_jobs=-1, verbose=-1)
    best.fit(X_train, y_train)
    return best


def calcula_metricas(y_true, y_pred, nome_modelo, rodada, posicao):
    """Retorna dict com todas as métricas de regressão para um modelo/rodada/posição."""
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    r2    = r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan
    rho, pval = spearmanr(y_true, y_pred)
    n     = len(y_true)
    # Top-3 precision: % dos 3 maiores pontuadores reais capturados no top-3 previsto
    if n >= 3:
        top3_real  = set(np.argsort(y_true)[-3:])
        top3_pred  = set(np.argsort(y_pred)[-3:])
        top3_prec  = len(top3_real & top3_pred) / 3
    else:
        top3_prec  = np.nan
    return {
        'modelo': nome_modelo, 'rodada': rodada, 'posicao': posicao,
        'n': n, 'rmse': rmse, 'mae': mae, 'r2': r2,
        'spearman': rho, 'spearman_p': pval, 'top3_precision': top3_prec
    }


print("Funções de modelo definidas ✓")


## 4. Orçamento Dinâmico

Simula um usuário real: começa com 130 cartoletas na rodada 5, e o patrimônio
evolui conforme as variações de preço dos atletas escalados e as trocas realizadas.


In [ ]:

def atualiza_patrimonio(time_anterior_ids, rodada_atual, players_df):
    """
    Calcula o patrimônio do time anterior na rodada atual,
    usando preco_num[atleta, rodada_atual] — que já reflete
    as variações decorrentes da rodada anterior.

    Retorna dict com:
        patrimonio_pos   : valor total do time após variações
        ganho_variacao   : variacao_num somada (patrimônio novo - patrimônio velho)
        df_variacao      : DataFrame detalhado por atleta
    """
    df_var = players_df[
        (players_df['atleta_id'].isin(time_anterior_ids)) &
        (players_df['rodada_id'] == rodada_atual)
    ][['atleta_id', 'apelido', 'preco_num', 'variacao_num']].copy()

    # Atletas que saíram do Cartola entre rodadas (raro, mas pode ocorrer)
    ids_encontrados = set(df_var['atleta_id'])
    ids_faltando    = set(time_anterior_ids) - ids_encontrados
    if ids_faltando:
        print(f"  [AVISO] Atletas não encontrados na rodada {rodada_atual}: {ids_faltando}")

    patrimonio_pos = df_var['preco_num'].sum()
    ganho_variacao = df_var['variacao_num'].sum()

    return {
        'patrimonio_pos': patrimonio_pos,
        'ganho_variacao': ganho_variacao,
        'df_variacao': df_var
    }


def calcula_n_trocas(time_anterior_ids, time_novo_ids):
    """Conta atletas que mudaram entre duas rodadas."""
    return len(set(time_anterior_ids) ^ set(time_novo_ids)) // 2


print("Funções de orçamento dinâmico definidas ✓")


## 5. Loop Walk-Forward Principal (Cenário A — Rodadas 5–18)

Este é o núcleo da avaliação retrospectiva. Para cada rodada:
1. Atualiza patrimônio com variações do time anterior
2. Calcula orçamento disponível
3. Treina modelos nas rodadas passadas
4. Prevê pontuações para atletas disponíveis
5. Otimiza escalação via PLI
6. Registra tudo no histórico


In [ ]:

# ─── Inicialização do estado do usuário simulado ──────────────────────────────

estado = {
    'time_escalado_ids': [],        # IDs dos 12 atletas do time atual
    'time_escalado_df':  None,      # DataFrame com detalhes do time
    'saldo_banco':       ORCAMENTO_INICIAL,
    'patrimonio':        0.0,
    'orcamento_total':   ORCAMENTO_INICIAL,
}

# Containers de output
historico_orcamento   = []   # 1 linha por rodada
historico_escalacoes  = []   # 1 linha por (atleta × rodada)
resultados_modelos    = []   # 1 linha por (modelo × posição × rodada)
feature_importances   = {}   # {(rodada, posicao, modelo): pd.Series}

rodadas_teste = list(range(MIN_TEST_ROUND, N_ROUNDS + 1))

print(f"Rodadas de teste: {rodadas_teste}")
print(f"Orçamento inicial: {ORCAMENTO_INICIAL} cartoletas\n")


In [ ]:

# ─── Loop principal ───────────────────────────────────────────────────────────

for t in rodadas_teste:
    print(f"\n{'='*60}")
    print(f"RODADA {t}")
    print(f"{'='*60}")

    # ── ETAPA 1: Atualiza patrimônio com variações do time anterior ────────────
    if estado['time_escalado_ids']:
        var_result = atualiza_patrimonio(
            estado['time_escalado_ids'], t, players
        )
        patrimonio_pos = var_result['patrimonio_pos']
        ganho_variacao = var_result['ganho_variacao']
        df_var_detalhe = var_result['df_variacao']
        print(f"  Patrimônio após variações: {patrimonio_pos:.2f} "
              f"(Δ {ganho_variacao:+.2f})")
    else:
        patrimonio_pos = 0.0
        ganho_variacao = 0.0
        df_var_detalhe = pd.DataFrame()

    # ── ETAPA 2: Calcula orçamento disponível ─────────────────────────────────
    orcamento_disponivel = estado['saldo_banco'] + patrimonio_pos
    print(f"  Orçamento disponível: {orcamento_disponivel:.2f} cartoletas")

    # ── ETAPA 3: Define treino e candidatos de teste ───────────────────────────
    df_treino = df[
        (df['rodada_id'] < t) &
        (df['rodada_id'] >= MIN_TRAIN_ROUND)
    ].copy()

    # Candidatos = atletas disponíveis na rodada t (usamos df_features como proxy;
    # em produção, buscar status real via API antes da rodada)
    df_teste = df[df['rodada_id'] == t].copy()

    # ── ETAPA 4: Treina modelos por posição e gera predições ──────────────────
    dfs_pred = []   # predições de todas as posições para o PLI

    for pos_id, pos_nome in POS_MAP.items():
        df_tr_pos  = df_treino[df_treino['posicao_id'] == pos_id].copy()
        df_te_pos  = df_teste[df_teste['posicao_id'] == pos_id].copy()

        if len(df_tr_pos) < 10 or len(df_te_pos) == 0:
            # Fallback: baseline se dados insuficientes
            df_te_pos['y_hat'] = baseline_predict(df_tr_pos, df_te_pos)
            df_te_pos['modelo_usado'] = 'baseline_fallback'
            dfs_pred.append(df_te_pos)
            continue

        X_train = df_tr_pos[FEATURE_COLS].fillna(0).values
        y_train = df_tr_pos['pontos_num'].values
        X_test  = df_te_pos[FEATURE_COLS].fillna(0).values
        y_true  = df_te_pos['pontos_num'].values

        preds_pos = {}

        # Baseline
        preds_pos['baseline'] = baseline_predict(df_tr_pos, df_te_pos).values

        # Ridge
        try:
            m_ridge = treina_ridge(X_train, y_train)
            preds_pos['ridge'] = m_ridge.predict(X_test)
        except Exception as e:
            preds_pos['ridge'] = preds_pos['baseline']
            print(f"  [ERRO Ridge {pos_nome}]: {e}")

        # XGBoost
        try:
            m_xgb = treina_xgboost(X_train, y_train, n_trials=30)
            preds_pos['xgboost'] = m_xgb.predict(X_test)
            # Feature importance
            fi = pd.Series(m_xgb.feature_importances_, index=FEATURE_COLS)
            feature_importances[(t, pos_nome, 'xgboost')] = fi
        except Exception as e:
            preds_pos['xgboost'] = preds_pos['baseline']
            print(f"  [ERRO XGBoost {pos_nome}]: {e}")

        # LightGBM
        try:
            m_lgb = treina_lightgbm(X_train, y_train, n_trials=30)
            preds_pos['lightgbm'] = m_lgb.predict(X_test)
            fi_lgb = pd.Series(m_lgb.feature_importances_, index=FEATURE_COLS)
            feature_importances[(t, pos_nome, 'lightgbm')] = fi_lgb
        except Exception as e:
            preds_pos['lightgbm'] = preds_pos['baseline']
            print(f"  [ERRO LightGBM {pos_nome}]: {e}")

        # Métricas para cada modelo
        for nome, y_pred in preds_pos.items():
            metricas = calcula_metricas(y_true, y_pred, nome, t, pos_nome)
            resultados_modelos.append(metricas)

        # Modelo principal para o PLI: XGBoost (melhor esperado)
        df_te_pos['y_hat'] = preds_pos['xgboost']
        df_te_pos['modelo_usado'] = 'xgboost'

        # Guarda predições de todos os modelos como colunas extras
        for nome, y_pred in preds_pos.items():
            df_te_pos[f'y_hat_{nome}'] = y_pred

        dfs_pred.append(df_te_pos)

    # ── ETAPA 5: Monta DataFrame de candidatos com predições ──────────────────
    df_candidatos = pd.concat(dfs_pred, ignore_index=True)

    # ── ETAPA 6: Otimizador PLI ───────────────────────────────────────────────
    escalacao = otimiza_escalacao(
        df_candidatos[['atleta_id', 'apelido', 'posicao_id', 'clube_id',
                        'preco_num', 'y_hat', 'pts_std_5r']],
        orcamento=orcamento_disponivel
    )
    otima = escalacao_otima(
        df_candidatos[['atleta_id', 'apelido', 'posicao_id', 'clube_id',
                        'preco_num', 'pontos_num', 'pts_std_5r']],
        orcamento=orcamento_disponivel
    )

    if escalacao is None:
        print(f"  [ERRO] PLI sem solução na rodada {t}. Pulando.")
        continue

    # ── ETAPA 7: Calcula pontuações reais da escalação ────────────────────────
    ids_escalados  = escalacao['atleta_id'].tolist()
    pts_obtidos    = df_candidatos[df_candidatos['atleta_id'].isin(ids_escalados)
                                   ]['pontos_num'].sum()
    pts_otimos     = otima['pontos_num'].sum() if otima is not None else np.nan
    custo_time     = escalacao['preco_num'].sum()
    eficiencia     = pts_obtidos / pts_otimos * 100 if pts_otimos else np.nan
    n_trocas       = calcula_n_trocas(estado['time_escalado_ids'], ids_escalados)

    print(f"  Time montado: {custo_time:.2f} cartoletas | "
          f"Pts previstos: {escalacao['y_hat'].sum():.2f} | "
          f"Pts reais: {pts_obtidos:.2f} | Ótimo: {pts_otimos:.2f} | "
          f"Eficiência: {eficiencia:.1f}%")

    # ── ETAPA 8: Atualiza estado do usuário ───────────────────────────────────
    novo_saldo_banco  = orcamento_disponivel - custo_time
    novo_patrimonio   = custo_time
    novo_orcamento    = novo_saldo_banco + novo_patrimonio

    print(f"  Saldo banco: {novo_saldo_banco:.2f} | "
          f"Patrimônio: {novo_patrimonio:.2f} | "
          f"Orçamento total: {novo_orcamento:.2f}")

    estado.update({
        'time_escalado_ids': ids_escalados,
        'time_escalado_df':  escalacao,
        'saldo_banco':       novo_saldo_banco,
        'patrimonio':        novo_patrimonio,
        'orcamento_total':   novo_orcamento,
    })

    # ── ETAPA 9: Registra histórico de orçamento ──────────────────────────────
    historico_orcamento.append({
        'rodada_id':             t,
        'orcamento_inicial_rodada': orcamento_disponivel,
        'custo_time_montado':    custo_time,
        'saldo_banco':           novo_saldo_banco,
        'patrimonio_pre':        patrimonio_pos - ganho_variacao,
        'patrimonio_pos':        patrimonio_pos,
        'ganho_variacao':        ganho_variacao,
        'orcamento_total':       novo_orcamento,
        'pontos_obtidos':        pts_obtidos,
        'pontos_otimos':         pts_otimos,
        'eficiencia_pct':        eficiencia,
        'n_trocas':              n_trocas,
    })

    # ── ETAPA 10: Registra histórico de escalações ────────────────────────────
    for _, row in escalacao.iterrows():
        atleta_id = row['atleta_id']
        y_hat_val = row['y_hat']
        pts_real  = df_candidatos.loc[
            df_candidatos['atleta_id'] == atleta_id, 'pontos_num'
        ].values
        pts_real  = pts_real[0] if len(pts_real) > 0 else np.nan

        historico_escalacoes.append({
            'rodada_id':       t,
            'atleta_id':       atleta_id,
            'apelido':         row['apelido'],
            'posicao_id':      row['posicao_id'],
            'posicao':         POS_MAP.get(row['posicao_id'], '?'),
            'clube_id':        row['clube_id'],
            'preco_entrada':   row['preco_num'],
            'variacao_rodada': df_var_detalhe.loc[
                df_var_detalhe['atleta_id'] == atleta_id, 'variacao_num'
            ].values[0] if atleta_id in df_var_detalhe['atleta_id'].values else np.nan,
            'pontos_previstos': y_hat_val,
            'pontos_obtidos':  pts_real,
            'erro_previsao':   pts_real - y_hat_val if not np.isnan(pts_real) else np.nan,
            'capitao':         row.get('capitao', False),
        })

print("\nWalk-forward concluído ✓")


## 6. Salva Outputs do Walk-Forward

In [ ]:

# DataFrames finais
df_historico_orc  = pd.DataFrame(historico_orcamento)
df_historico_esc  = pd.DataFrame(historico_escalacoes)
df_resultados     = pd.DataFrame(resultados_modelos)

# Salva CSVs
df_historico_orc.to_csv(PATH_OUT + 'historico_orcamento.csv',  index=False)
df_historico_esc.to_csv(PATH_OUT + 'historico_escalacoes.csv', index=False)
df_resultados.to_csv(   PATH_OUT + 'resultados_walkforward.csv', index=False)

# Feature importances (XGBoost)
fi_records = []
for (rodada, posicao, modelo), fi_series in feature_importances.items():
    for feat, imp in fi_series.items():
        fi_records.append({'rodada': rodada, 'posicao': posicao,
                           'modelo': modelo, 'feature': feat, 'importance': imp})
pd.DataFrame(fi_records).to_csv(PATH_OUT + 'feature_importances.csv', index=False)

print("Outputs salvos em", PATH_OUT)
print(f"  historico_orcamento.csv   : {df_historico_orc.shape}")
print(f"  historico_escalacoes.csv  : {df_historico_esc.shape}")
print(f"  resultados_walkforward.csv: {df_resultados.shape}")
print(f"  feature_importances.csv   : {len(fi_records)} linhas")


In [ ]:

# Preview dos resultados
print("\n=== Histórico de Orçamento ===")
cols_exibir = ['rodada_id', 'orcamento_inicial_rodada', 'custo_time_montado',
               'ganho_variacao', 'orcamento_total', 'pontos_obtidos',
               'pontos_otimos', 'eficiencia_pct']
print(df_historico_orc[cols_exibir].to_string(index=False, float_format='{:.2f}'.format))

print("\n=== Resumo por Modelo ===")
print(df_resultados.groupby('modelo')[['rmse','mae','spearman','top3_precision']].mean().round(3))


## 7. Visualizações para o TCC

Gera as figuras da Seção 6 do TCC a partir dos outputs do walk-forward.


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Desempenho do Sistema — Walk-Forward (Rodadas 5–18)', fontsize=14, fontweight='bold')

# ── A: Evolução patrimonial ────────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(df_historico_orc['rodada_id'], df_historico_orc['orcamento_total'],
        marker='o', color='#003366', linewidth=2, label='Sistema (XGBoost)')
ax.axhline(ORCAMENTO_INICIAL, color='gray', linestyle='--', alpha=0.7, label='Inicial (130¢)')
ax.fill_between(df_historico_orc['rodada_id'],
                ORCAMENTO_INICIAL, df_historico_orc['orcamento_total'],
                alpha=0.15, color='#003366')
ax.set_xlabel('Rodada'); ax.set_ylabel('Cartoletas')
ax.set_title('(A) Evolução Patrimonial', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)

# ── B: Decomposição ganho por rodada ──────────────────────────────────────────
ax2 = axes[0, 1]
x   = df_historico_orc['rodada_id']
ax2.bar(x, df_historico_orc['ganho_variacao'], color='#1a7a4a', alpha=0.8,
        label='Ganho por variação de preço')
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xlabel('Rodada'); ax2.set_ylabel('Δ Cartoletas')
ax2.set_title('(B) Ganho/Perda por Variação de Preço por Rodada', fontweight='bold')
ax2.legend(); ax2.grid(alpha=0.3)

# ── C: Pontuação por rodada (sistema vs ótimo) ────────────────────────────────
ax3 = axes[1, 0]
ax3.plot(x, df_historico_orc['pontos_obtidos'],
         marker='o', color='#003366', linewidth=2, label='Escalação do Sistema')
ax3.plot(x, df_historico_orc['pontos_otimos'],
         marker='s', color='#C0392B', linewidth=2, linestyle='--', label='Escalação Ótima')
ax3.set_xlabel('Rodada'); ax3.set_ylabel('Pontuação Total')
ax3.set_title('(C) Pontuação por Rodada: Sistema vs. Ótimo', fontweight='bold')
ax3.legend(); ax3.grid(alpha=0.3)

# ── D: Eficiência por rodada ──────────────────────────────────────────────────
ax4 = axes[1, 1]
colors_eff = ['#1a7a4a' if e >= 70 else '#E67E22' if e >= 50 else '#C0392B'
              for e in df_historico_orc['eficiencia_pct']]
ax4.bar(x, df_historico_orc['eficiencia_pct'], color=colors_eff, alpha=0.85)
ax4.axhline(df_historico_orc['eficiencia_pct'].mean(), color='black',
            linestyle='--', linewidth=1.5,
            label=f"Média: {df_historico_orc['eficiencia_pct'].mean():.1f}%")
ax4.set_xlabel('Rodada'); ax4.set_ylabel('Eficiência (%)')
ax4.set_ylim(0, 110)
ax4.set_title('(D) Eficiência da Escalação (Obtido / Ótimo × 100%)', fontweight='bold')
ax4.legend(); ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PATH_OUT + 'fig_desempenho_sistema.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva ✓")


In [ ]:

# ── Comparação de modelos: RMSE e Spearman por posição ────────────────────────
pivot_rmse = df_resultados.pivot_table(
    index='posicao', columns='modelo', values='rmse', aggfunc='mean'
).round(3)
pivot_spe  = df_resultados.pivot_table(
    index='posicao', columns='modelo', values='spearman', aggfunc='mean'
).round(3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Comparação de Modelos por Posição', fontsize=13, fontweight='bold')

pivot_rmse.plot(kind='bar', ax=axes[0], colormap='tab10', edgecolor='white')
axes[0].set_title('RMSE por Posição (menor = melhor)', fontweight='bold')
axes[0].set_xlabel(''); axes[0].set_ylabel('RMSE (pts)')
axes[0].legend(title='Modelo', bbox_to_anchor=(1, 1))
axes[0].tick_params(axis='x', rotation=30)

pivot_spe.plot(kind='bar', ax=axes[1], colormap='tab10', edgecolor='white')
axes[1].set_title('Spearman ρ por Posição (maior = melhor)', fontweight='bold')
axes[1].set_xlabel(''); axes[1].set_ylabel('Spearman ρ')
axes[1].legend(title='Modelo', bbox_to_anchor=(1, 1))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(PATH_OUT + 'fig_comparacao_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTabela RMSE por modelo e posição:")
print(pivot_rmse.to_string())
print("\nTabela Spearman por modelo e posição:")
print(pivot_spe.to_string())


In [ ]:

# ── Feature Importance média por posição (XGBoost) ────────────────────────────
df_fi = pd.read_csv(PATH_OUT + 'feature_importances.csv')
df_fi_xgb = df_fi[df_fi['modelo'] == 'xgboost']

posicoes_plot = ['Atacante', 'Meia', 'Goleiro', 'Lateral']
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Feature Importance — XGBoost por Posição (Top 15)', fontsize=13, fontweight='bold')

for ax, pos in zip(axes.flat, posicoes_plot):
    fi_pos = (df_fi_xgb[df_fi_xgb['posicao'] == pos]
              .groupby('feature')['importance'].mean()
              .sort_values(ascending=False)
              .head(15))
    fi_pos[::-1].plot(kind='barh', ax=ax, color='#003366', alpha=0.8)
    ax.set_title(f'{pos}', fontweight='bold')
    ax.set_xlabel('Importância média')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PATH_OUT + 'fig_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Feature importance salva ✓")


In [ ]:

# ── Scatter: pontos previstos vs reais por posição ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Pontos Previstos vs. Reais por Posição (Walk-Forward)',
             fontsize=13, fontweight='bold')

for ax, (pos_id, pos_nome) in zip(axes.flat, POS_MAP.items()):
    sub = df_historico_esc[df_historico_esc['posicao'] == pos_nome]
    if sub.empty:
        ax.set_visible(False); continue
    ax.scatter(sub['pontos_previstos'], sub['pontos_obtidos'],
               alpha=0.4, s=20, color='#003366')
    lim = max(sub['pontos_previstos'].max(), sub['pontos_obtidos'].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'r--', alpha=0.6, linewidth=1)
    rho = sub[['pontos_previstos','pontos_obtidos']].corr(method='spearman').iloc[0,1]
    mae = (sub['pontos_previstos'] - sub['pontos_obtidos']).abs().mean()
    ax.set_title(f'{pos_nome}  |  ρ={rho:.2f}, MAE={mae:.2f}', fontweight='bold')
    ax.set_xlabel('Previsto (pts)'); ax.set_ylabel('Real (pts)')
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig(PATH_OUT + 'fig_previsto_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 8. Previsão do Segundo Turno (Cenário B — Rodadas 19–36)

Treina nos dados completos (rodadas 1–18) e gera escalações incrementalmente
conforme as rodadas do segundo turno são disputadas.

**Como usar:**
1. Após cada rodada R disputada, rode o repositório para atualizar `players.csv`,
   `games.csv` e regenerar `df_features.csv` com as rodadas 1 até R.
2. Execute a célula abaixo para gerar a escalação da rodada R+1.


In [ ]:

def gera_escalacao_segundo_turno(
    df_features_atualizado,
    players_atualizado,
    rodada_prev,
    estado_usuario,
    path_out=PATH_OUT
):
    """
    Gera a escalação para a rodada rodada_prev usando todos os dados disponíveis.

    Parâmetros
    ----------
    df_features_atualizado : DataFrame — df_features com todas as rodadas até rodada_prev - 1
    players_atualizado     : DataFrame — players.csv atualizado
    rodada_prev            : int       — rodada a prever (ex: 19)
    estado_usuario         : dict      — estado atual (saldo_banco, time_escalado_ids, etc.)
    path_out               : str       — pasta de saída

    Retorna
    -------
    escalacao : DataFrame com os 12 atletas
    estado    : dict com estado atualizado
    """
    t = rodada_prev
    print(f"\nGerando escalação para rodada {t}...")

    # Atualiza patrimônio se já há um time escalado
    rodada_ultima = t - 1
    if estado_usuario['time_escalado_ids'] and rodada_ultima in players_atualizado['rodada_id'].values:
        var_result = atualiza_patrimonio(
            estado_usuario['time_escalado_ids'], rodada_ultima, players_atualizado
        )
        patrimonio_pos = var_result['patrimonio_pos']
        ganho_variacao = var_result['ganho_variacao']
    else:
        patrimonio_pos = estado_usuario['patrimonio']
        ganho_variacao = 0.0

    orcamento_disponivel = estado_usuario['saldo_banco'] + patrimonio_pos
    print(f"  Orçamento disponível: {orcamento_disponivel:.2f} cartoletas")

    # Dados de treino: todas as rodadas disponíveis
    df_treino = df_features_atualizado[
        df_features_atualizado['rodada_id'] >= MIN_TRAIN_ROUND
    ].copy()

    # Candidatos: atletas disponíveis para a rodada_prev
    # Em produção: buscar via API os atletas com status Provável para a rodada t
    # Aqui usamos o df_features se já tiver a rodada t; senão, usar a rodada mais recente
    if t in df_features_atualizado['rodada_id'].values:
        df_cand = df_features_atualizado[df_features_atualizado['rodada_id'] == t].copy()
    else:
        # Fallback: usa atletas da última rodada disponível com preços atualizados
        ultima_rod = df_features_atualizado['rodada_id'].max()
        df_cand    = df_features_atualizado[df_features_atualizado['rodada_id'] == ultima_rod].copy()
        print(f"  [AVISO] Rodada {t} não encontrada; usando rodada {ultima_rod} como proxy.")

    dfs_pred = []
    for pos_id, pos_nome in POS_MAP.items():
        df_tr_pos = df_treino[df_treino['posicao_id'] == pos_id].copy()
        df_te_pos = df_cand[df_cand['posicao_id'] == pos_id].copy()

        if len(df_tr_pos) < 10 or len(df_te_pos) == 0:
            df_te_pos['y_hat'] = baseline_predict(df_tr_pos, df_te_pos)
        else:
            X_train = df_tr_pos[FEATURE_COLS].fillna(0).values
            y_train = df_tr_pos['pontos_num'].values
            X_test  = df_te_pos[FEATURE_COLS].fillna(0).values
            try:
                m = treina_xgboost(X_train, y_train, n_trials=30)
                df_te_pos['y_hat'] = m.predict(X_test)
            except:
                df_te_pos['y_hat'] = baseline_predict(df_tr_pos, df_te_pos)

        dfs_pred.append(df_te_pos)

    df_cand_pred = pd.concat(dfs_pred, ignore_index=True)

    escalacao = otimiza_escalacao(
        df_cand_pred[['atleta_id', 'apelido', 'posicao_id', 'clube_id',
                       'preco_num', 'y_hat', 'pts_std_5r']],
        orcamento=orcamento_disponivel
    )

    if escalacao is None:
        print("  [ERRO] PLI sem solução.")
        return None, estado_usuario

    custo = escalacao['preco_num'].sum()
    novo_estado = {
        'time_escalado_ids': escalacao['atleta_id'].tolist(),
        'time_escalado_df':  escalacao,
        'saldo_banco':       orcamento_disponivel - custo,
        'patrimonio':        custo,
        'orcamento_total':   orcamento_disponivel,   # atualizado após a rodada
    }

    # Salva a escalação desta rodada
    escalacao_out = escalacao.copy()
    escalacao_out['rodada_prevista'] = t
    escalacao_out['orcamento_disponivel'] = orcamento_disponivel
    escalacao_out['custo_time'] = custo
    escalacao_out['ganho_variacao_anterior'] = ganho_variacao

    fname = path_out + f'escalacao_rodada_{t}.csv'
    escalacao_out.to_csv(fname, index=False)
    print(f"  Escalação salva em {fname}")
    print(f"  Custo: {custo:.2f} | Pts previstos: {escalacao['y_hat'].sum():.2f}")
    print(escalacao[['apelido', 'posicao_id', 'preco_num', 'y_hat', 'capitao']]
          .sort_values('posicao_id').to_string(index=False))

    return escalacao, novo_estado


# ── Exemplo de uso ────────────────────────────────────────────────────────────
# Após rodar o walk-forward, 'estado' contém o estado ao fim da rodada 18.
# Para gerar a escalação da rodada 19:
#
# df_features_19 = pd.read_csv('df_features.csv')   # regenerado pelo repositório
# players_19     = pd.read_csv('players.csv')
#
# escalacao_19, estado = gera_escalacao_segundo_turno(
#     df_features_19, players_19, rodada_prev=19, estado_usuario=estado
# )

print("Função de segundo turno definida ✓")
print("Para usar: descomente o bloco de exemplo acima e ajuste os caminhos.")


## 9. Tabelas Resumo para o TCC

In [ ]:

# ── Tabela 6.1: Desempenho dos Modelos (média walk-forward) ───────────────────
print("=== Tabela 6.1 — Desempenho dos Modelos (walk-forward, rodadas 5–18) ===\n")
tab_modelos = df_resultados.groupby('modelo').agg(
    RMSE   =('rmse',           'mean'),
    MAE    =('mae',            'mean'),
    R2     =('r2',             'mean'),
    Spearman=('spearman',      'mean'),
    Top3Prec=('top3_precision','mean'),
    N      =('n',              'sum'),
).round(3).sort_values('MAE')
print(tab_modelos.to_string())

# ── Tabela 6.2: Qualidade das Escalações ─────────────────────────────────────
print("\n=== Tabela 6.2 — Qualidade das Escalações (walk-forward) ===\n")
tab_esc = df_historico_orc[[
    'rodada_id', 'orcamento_total', 'pontos_obtidos',
    'pontos_otimos', 'eficiencia_pct', 'ganho_variacao', 'n_trocas'
]].copy()
tab_esc.loc['Média'] = tab_esc.mean(numeric_only=True)
tab_esc.loc['Média', 'rodada_id'] = '--'
print(tab_esc.to_string(index=False, float_format='{:.2f}'.format))

# ── Salva tabelas como CSV para colar no LaTeX ───────────────────────────────
tab_modelos.to_csv(PATH_OUT + 'tabela_modelos.csv')
df_historico_orc.to_csv(PATH_OUT + 'tabela_escalacoes.csv', index=False)
print("\nTabelas salvas em", PATH_OUT)


---
## Checklist de Implementação

### Cenário A (Walk-Forward, Rodadas 5–18)
- [ ] Seção 0: imports e configuração
- [ ] Seção 1: carrega dados e pré-processa h2h_pts_avg
- [ ] Seção 2: testa otimizador PLI com dados sintéticos
- [ ] Seção 3: valida modelos em uma posição (Atacante) antes de escalar
- [ ] Seção 4: valida lógica de orçamento dinâmico manualmente (rodadas 5–7)
- [ ] Seção 5: roda loop completo (rodadas 5–18, todos os modelos e posições)
- [ ] Seção 6: salva os 4 CSVs de output
- [ ] Seção 7: gera as 4 figuras e as salva em `outputs/`
- [ ] Seção 9: gera tabelas para colar no LaTeX

### Cenário B (Segundo Turno, Rodadas 19–36)
- [ ] Coletar dados da rodada R via repositório
- [ ] Regenerar `df_features.csv` com rodadas 1 a R
- [ ] Chamar `gera_escalacao_segundo_turno()` para a rodada R+1
- [ ] Após a rodada R+1: registrar pontuações reais e atualizar `estado`
- [ ] Repetir até a rodada 36

### Outputs Gerados
| Arquivo | Descrição |
|---|---|
| `historico_orcamento.csv` | 1 linha/rodada — patrimônio, pontos, eficiência |
| `historico_escalacoes.csv` | 1 linha/atleta×rodada — previsão, real, erro |
| `resultados_walkforward.csv` | 1 linha/modelo×posição×rodada — métricas |
| `feature_importances.csv` | Importância de cada feature por posição |
| `escalacao_rodada_R.csv` | Escalação sugerida para cada rodada do 2º turno |
| `fig_desempenho_sistema.png` | Evolução patrimonial e eficiência |
| `fig_comparacao_modelos.png` | RMSE e Spearman por modelo e posição |
| `fig_feature_importance.png` | Top-15 features por posição (XGBoost) |
| `fig_previsto_vs_real.png` | Scatter previsto vs. real por posição |
